# Embedding

In [ ]:
import numpy as np
from langchain_ollama import OllamaEmbeddings

EMBED_MODEL = "qwen3-embedding:0.6b"
embedding_model = OllamaEmbeddings(model=EMBED_MODEL)

msg = embedding_model.embed_query("혼자 사는 어르신 돌봄")

print("벡터 길이: ", len(msg))
print("앞 5개: ", msg[:5])

벡터 길이:  1024
앞 5개:  [-0.040879592, 0.009949414, -0.013987224, 0.0450653, 0.013824958]


## 청크 파일 읽기

In [5]:
from langchain_community.document_loaders import JSONLoader

def metadata_func(record: dict, metadata: dict) -> dict:
    for key, value in record.items():
        if key != "text":
            metadata[key] = value
    return metadata

loader = JSONLoader(
    file_path="./data/sample_policy_chunks.jsonl",
    jq_schema=".",
    content_key="text",
    json_lines=True,
    metadata_func=metadata_func,
)

documents = loader.load()
print(f"로드된 문서 수: {len(documents)}")
print(documents[0].page_content)
print(documents[0].metadata)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_20700\750381474.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


로드된 문서 수: 6
[경기 수원시] 독거노인 생활지원 서비스. 대상: 수원시에 사는 65세 이상 혼자 사는 어르신. 내용: 생활관리사가 안부 확인과 생활 지원을 합니다. 신청: 주소지 행정복지센터.
{'source': 'C:\\dev\\SKN36-3rd-3Team\\notebooks\\qdrant\\data\\sample_policy_chunks.jsonl', 'seq_num': 1, 'chunk_id': 'POL-41110-0001-c0', 'policy_id': 'POL-41110-0001', 'document_id': None, 'section_id': None, 'region_code': '41110', 'region_name': '경기도 수원시', 'region_scope': 'sigungu', 'year': 2026, 'source_type': 'local_api', 'source_name': '지자체복지서비스 API', 'page': None, 'source_url': 'https://example.go.kr/policy/41110-0001', 'review_status': 'approved'}


## 전체 청크 임베딩하기

In [9]:
texts = [doc.page_content for doc in documents]
vectors = embedding_model.embed_documents(texts)

print(len(vectors), "개 /", len(vectors[0]), "차원")

6 개 / 1024 차원


## 벡터 파일로 저장 

In [10]:
import json

with open("./data/policy_embeddings.jsonl", "w", encoding="utf-8") as f:
    for doc, vec in zip(documents, vectors):
        row = {
            "chunk_id": doc.metadata["chunk_id"],
            "policy_id": doc.metadata.get("policy_id"),
            "document_id": doc.metadata.get("document_id"),
            "model": EMBED_MODEL,
            "dim": len(vec),
            "vector": vec,
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("저장 완료")

저장 완료
